# Озеро ODS: март-дубли INT и май-объём trx

Две независимые проверки. Не смешивать в одном тикете.

**Блок 1** — тот же SQL, которым уже получили **666 739** дублей: секция **5f** в `excel_march_2026_chod_finrez_dip.ipynb` (`MEM_LIMIT=16g`, один месяц, периметр витрины).

**Блок 2** — объём `trx_cnt` / `trx_sum` в ODS за апр–май–июн (55% vs Excel этим SQL не считается).


In [ ]:
from calendar import monthrange

from IPython.display import display
import pandas as pd
from rail_connectors.connection import connect

MEM_LIMIT = '16g'

if 'imp' not in globals() or imp is None:
    imp = connect(
        to='IMPALA',
        extra_options={'db': 'sandbox_ai'},
        driver_args={'tez.queue.name': 'ai'},
        kerberos={
            'keytab_path': '/home/jovyan/test_requests/tech.keytab',
            'use_credentials': True,
            'update_keytab': True,
        },
        user_params={'user_name': 'Shestopalov-VYur'},
    )
    imp._init_connection()
    print('Impala connected')
else:
    print('Reuse existing imp')


def run_sql(sql, title=None):
    if title:
        print(title)
    with imp:
        imp.execute(f'set MEM_LIMIT={MEM_LIMIT}')
        df = imp.fetch(sql)
    display(df)
    return df


## 1) Дубли `n_amt_fee` — как в 5f (уже давало 666 739)

Не сырой `scd1_trx` × вся `trx_int`. Сначала узкие ключи витрины, потом fee только по ним:

1. `scd1_base24_fiids` — RSHB
2. `scd1_agreements` — SA, живые в месяце
3. `scd1_trx` — SA/S01, не reversed, `d_trx_orig` в месяце
4. `scd1_trx_acq` — есть `n_agr` из SA
5. `scd1_trx_int` — живые строки, `count(*)>1` и `count(distinct n_amt_fee)=1`

Один месяц за раз. Ожидание: март `multi_same_fee_dup` ≈ **666 739**, avg rows = 2; февраль ≈ 1; апрель ≈ 0.


In [ ]:
def month_bounds(ym):
    y, m = map(int, ym.split('-'))
    return f'{ym}-01', f'{ym}-{monthrange(y, m)[1]:02d}'


def cte_trx_keys(ms, me):
    # Как excel_march_2026_chod_finrez_dip.ipynb / секция 5f
    return f"""
    fiid_rshb as (
      select distinct cast(fa.c_fiid as string) as c_fiid
      from ods_alpha.scd1_base24_fiids fa
      where coalesce(cast(fa.c_fiid_grp as string), 'UNKNOWN') = 'RSHB'
    ),
    sa_agr as (
      select distinct cast(a.n_agr as string) as n_agr
      from ods_alpha.scd1_agreements a
      where upper(trim(cast(a.acq_class as string))) = 'SA'
        and cast(a.d_valid_from as date) <= cast('{me}' as date)
        and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{ms}' as date))
        and coalesce(a.ods_deleted_flg, '0') <> '1'
    ),
    trx_base as (
      select cast(t.n_trx as string) as n_trx
      from ods_alpha.scd1_trx t
      join fiid_rshb fr on fr.c_fiid = cast(t.c_fiid_acq as string)
      where cast(t.d_trx_orig as timestamp) >= cast('{ms}' as timestamp)
        and cast(t.d_trx_orig as timestamp) < cast(date_add(cast('{me}' as date), 1) as timestamp)
        and t.c_nter is not null
        and coalesce(t.ods_deleted_flg, '0') <> '1'
        and t.c_trx_class = 'SA'
        and t.c_trx_type = 'S01'
        and coalesce(t.cf_trx_stat, '') <> 'R'
      group by cast(t.n_trx as string)
    ),
    ta as (
      select cast(a.n_trx as string) as n_trx
      from ods_alpha.scd1_trx_acq a
      join trx_base tb on tb.n_trx = cast(a.n_trx as string)
      join sa_agr ss on ss.n_agr = cast(a.n_agr as string)
      where coalesce(a.ods_deleted_flg, '0') <> '1'
      group by cast(a.n_trx as string)
    )
    """


def sql_multi_class(ym):
    ms, me = month_bounds(ym)
    return f"""
    with {cte_trx_keys(ms, me)},
    int_alive as (
      select
        cast(i.n_trx as string) as n_trx,
        coalesce(cast(i.n_amt_fee as double), 0.0) as n_amt_fee
      from ods_alpha.scd1_trx_int i
      join ta k on k.n_trx = cast(i.n_trx as string)
      where coalesce(i.ods_deleted_flg, '0') <> '1'
    ),
    per_trx as (
      select
        n_trx,
        count(*) as int_rows,
        count(distinct cast(n_amt_fee as string)) as distinct_fee_values,
        sum(n_amt_fee) as fee_sum,
        max(n_amt_fee) as fee_max
      from int_alive
      group by n_trx
    ),
    multi as (
      select * from per_trx where int_rows > 1
    )
    select
      '{ym}' as report_month,
      count(*) as multi_trx_cnt,
      sum(case when distinct_fee_values = 1 then 1 else 0 end) as multi_same_fee_dup,
      sum(case when distinct_fee_values > 1 then 1 else 0 end) as multi_different_fees,
      avg(int_rows) as avg_rows_on_multi,
      sum(case when distinct_fee_values = 1 then fee_sum - fee_max else 0 end) as extra_fee_from_same_fee_dups
    from multi
    """


DUP_MONTHS = ['2026-02', '2026-03', '2026-04']
dup_parts = []
for ym in DUP_MONTHS:
    part = run_sql(sql_multi_class(ym), f'=== 1. Дубли trx_int (5f) {ym} ===')
    if part is not None and len(part):
        dup_parts.append(part)

march_by_month = pd.concat(dup_parts, ignore_index=True) if dup_parts else pd.DataFrame()
print('=== свод ===')
display(march_by_month)


Sample двух живых строк на один мартовский `n_trx` (тот же периметр `ta`).


In [ ]:
ms, me = month_bounds('2026-03')
sql_march_sample = f"""
with {cte_trx_keys(ms, me)},
multi as (
  select cast(i.n_trx as string) as n_trx
  from ods_alpha.scd1_trx_int i
  join ta k on k.n_trx = cast(i.n_trx as string)
  where coalesce(i.ods_deleted_flg, '0') <> '1'
  group by cast(i.n_trx as string)
  having count(*) > 1
     and count(distinct cast(i.n_amt_fee as string)) = 1
  limit 8
)
select
  cast(i.n_trx as string) as n_trx,
  cast(i.n_amt_fee as double) as n_amt_fee,
  cast(i.ods_deleted_flg as string) as ods_deleted_flg,
  cast(i.ods_op_type as string) as ods_op_type
from ods_alpha.scd1_trx_int i
join multi m on m.n_trx = cast(i.n_trx as string)
where coalesce(i.ods_deleted_flg, '0') <> '1'
order by 1, 2
"""

march_sample = run_sql(sql_march_sample, '=== 1b. Sample дублей март (периметр 5f) ===')


## 2) Май: количество и сумма транзакций в ODS

Если май здесь как апрель/июнь, а сверка с Excel 55% — смотреть Excel. Если май просел здесь — озеро.


In [ ]:
sql_may_trx_simple = r"""
select
  trunc(to_date(cast(t.d_trx_orig as timestamp)), 'MM') as trx_month,
  count(*) as rows_cnt,
  count(distinct t.n_trx) as trx_cnt,
  sum(cast(t.n_amt_src as double)) as trx_sum
from ods_alpha.scd1_trx t
where coalesce(t.ods_deleted_flg, '0') <> '1'
  and t.c_trx_class = 'SA'
  and t.c_trx_type = 'S01'
  and t.c_nter is not null
  and coalesce(t.cf_trx_stat, '') <> 'R'
  and cast(t.d_trx_orig as timestamp) >= timestamp '2026-04-01 00:00:00'
  and cast(t.d_trx_orig as timestamp) <  timestamp '2026-07-01 00:00:00'
group by 1
order by 1
"""

may_simple = run_sql(
    sql_may_trx_simple,
    '=== 2. ODS trx_cnt / trx_sum (только scd1_trx) ===',
)


Как витрина: RSHB + `trx_acq` на периметре месяца (как `ta` в 5f).


In [ ]:
def sql_may_volume_mart(ym):
    ms, me = month_bounds(ym)
    return f"""
    with {cte_trx_keys(ms, me)},
    trx_amt as (
      select
        cast(t.n_trx as string) as n_trx,
        max(cast(t.n_amt_src as double)) as n_amt_src
      from ods_alpha.scd1_trx t
      join ta k on k.n_trx = cast(t.n_trx as string)
      group by cast(t.n_trx as string)
    )
    select
      '{ym}' as report_month,
      count(*) as trx_cnt,
      sum(n_amt_src) as trx_sum
    from trx_amt
    """


vol_parts = []
for ym in ['2026-04', '2026-05', '2026-06']:
    part = run_sql(sql_may_volume_mart(ym), f'=== 2b. Объём как витрина {ym} ===')
    if part is not None and len(part):
        vol_parts.append(part)

may_mart = pd.concat(vol_parts, ignore_index=True) if vol_parts else pd.DataFrame()
print('=== свод апр / май / июн ===')
display(may_mart)


Май по дням: нет ли обрыва загрузки.


In [ ]:
sql_may_daily = r"""
select
  to_date(cast(t.d_trx_orig as timestamp)) as trx_dt,
  count(distinct t.n_trx) as trx_cnt,
  sum(cast(t.n_amt_src as double)) as trx_sum
from ods_alpha.scd1_trx t
where coalesce(t.ods_deleted_flg, '0') <> '1'
  and t.c_trx_class = 'SA'
  and t.c_trx_type = 'S01'
  and t.c_nter is not null
  and coalesce(t.cf_trx_stat, '') <> 'R'
  and cast(t.d_trx_orig as timestamp) >= timestamp '2026-05-01 00:00:00'
  and cast(t.d_trx_orig as timestamp) <  timestamp '2026-06-01 00:00:00'
group by 1
order by 1
"""

may_daily = run_sql(sql_may_daily, '=== 2c. Май по дням ===')
if may_daily is not None and len(may_daily):
    print(
        'дней с данными:', len(may_daily),
        '| min', may_daily['trx_dt'].min(),
        '| max', may_daily['trx_dt'].max(),
    )


## Что отправить озерщикам

**Тикет 1 — март.** SQL ячейки 1 (как 5f). Таблица `ods_alpha.scd1_trx_int`.

**Тикет 2 — май.** Ячейки 2 / 2b. Цифру 55% не ставить.
